# ACDC cardiac MRI segmentation -- training

Trains a 2D U-Net (via `segmentation-models-pytorch`) with four different
encoder backbones on the ACDC dataset, and writes one results file
(`results/all_results.json`) that the evaluation notebook reads from.

**Encoders compared:** ResNet18, VGG16, MobileNetV2, ResNet50.

Run this notebook on a GPU (Kaggle / Colab / local CUDA box). It downloads
the ACDC dataset via `kagglehub`, trains each encoder, and saves:
- one checkpoint per encoder to `checkpoints/<encoder>_best.pth`
- one shared results file to `results/all_results.json`


## Setup

In [ ]:
%pip install -q lightning nibabel segmentation-models-pytorch kagglehub

import sys
sys.path.insert(0, "..")  # so `import src` works from notebooks/

import os
import json

import torch
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from src.dataset import ACDC2DDataModule
from src.model import LitUNet2D
from src.benchmark import measure_batch_latency, model_size_mb, load_results, save_results, update_result


In [ ]:
import kagglehub

data_path = kagglehub.dataset_download("samdazel/automated-cardiac-diagnosis-challenge-miccai17")
acdc_root = f"{data_path}/database/training"

print("Data downloaded to:", data_path)


In [ ]:
# ---- Config ----
ENCODERS = ["resnet18", "vgg16", "mobilenet_v2", "resnet50"]
MAX_EPOCHS = 50
BATCH_SIZE = 8
LR = 1e-3
SEED = 42

CHECKPOINT_DIR = "../checkpoints"
RESULTS_PATH = "../results/all_results.json"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)


## Data

In [ ]:
pl.seed_everything(SEED, workers=True)

dm = ACDC2DDataModule(acdc_root, batch_size=BATCH_SIZE, num_workers=2, seed=SEED)
dm.setup()

print(f"train slices: {len(dm.train_ds)}")
print(f"val slices:   {len(dm.val_ds)}")
print(f"test slices:  {len(dm.test_ds)}")


## Train all encoders

For each encoder: train with early stopping on `val_dice`, reload the best
checkpoint, evaluate on the test set, measure batch-level inference latency
and model size, then merge everything into `results/all_results.json`.

Note: this writes the **batch-8** inference latency
(`inference_ms_batch8_mean`). Per-slice (batch_size=1) latency, which is
the number used for the headline latency comparison figure, is measured
separately in `notebooks/02_evaluate.ipynb` -- that measurement needs a
saved checkpoint to load, so it happens after training, not during it.


In [ ]:
results = load_results(RESULTS_PATH)

for encoder in ENCODERS:
    print(f"\n{'='*40}")
    print(f"Training: {encoder}")
    print(f"{'='*40}")

    model = LitUNet2D(lr=LR, encoder_name=encoder)

    early_stop = EarlyStopping(monitor="val_dice", mode="max", patience=20, verbose=True)
    checkpoint = ModelCheckpoint(
        monitor="val_dice", mode="max", save_top_k=1,
        dirpath=CHECKPOINT_DIR,
        filename=f"{encoder}-acdc-{{epoch:02d}}-{{val_dice:.4f}}",
    )
    trainer = Trainer(
        max_epochs=MAX_EPOCHS, callbacks=[early_stop, checkpoint],
        log_every_n_steps=5, enable_progress_bar=True, logger=False,
    )

    trainer.fit(model, dm)

    best_model = LitUNet2D.load_from_checkpoint(checkpoint.best_model_path)
    best_model = best_model.to("cuda")

    # Save the plain state_dict too -- this is what evaluate.ipynb loads,
    # since it's lighter than the full Lightning checkpoint.
    final_ckpt_path = os.path.join(CHECKPOINT_DIR, f"{encoder}_best.pth")
    torch.save(best_model.state_dict(), final_ckpt_path)

    best_model.eval()
    batch_latency_ms = measure_batch_latency(best_model, dm.val_dataloader(), device="cuda", n_batches=50)

    test_res = trainer.test(best_model, datamodule=dm, verbose=False)
    size_mb = model_size_mb(best_model)

    update_result(
        results, encoder,
        dice=round(test_res[0].get("test_dice", -1), 4),
        iou=round(test_res[0].get("test_miou", -1), 4),
        inference_ms_batch8_mean=round(batch_latency_ms, 2),
        model_size_mb=size_mb,
    )

    save_results(results, RESULTS_PATH)  # save after every encoder, not just at the end
    print(json.dumps(results[encoder], indent=2))

print("\nALL TRAINING RUNS COMPLETE")
print(json.dumps(results, indent=2))


## Next step

Run `notebooks/02_evaluate.ipynb` to:
- measure per-slice (batch_size=1) inference latency
- compute per-class Dice (LV / RV / Myocardium)
- generate the qualitative segmentation comparison figure
- generate the metrics bar chart

Both notebooks read and write `results/all_results.json` -- it is the only
place these numbers live.
